In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('clockify_unificado_mascarado.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1319 entries, 0 to 1318
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   User                 1319 non-null   str    
 1   Tags                 1319 non-null   str    
 2   Billable             1319 non-null   str    
 3   Start Date           1319 non-null   str    
 4   Start Time           1319 non-null   str    
 5   End Date             1319 non-null   str    
 6   End Time             1319 non-null   str    
 7   Duration (h)         1319 non-null   str    
 8   Duration (decimal)   1319 non-null   float64
 9   Date of creation     1319 non-null   str    
 10  cliente_mascarado    1314 non-null   str    
 11  sistema_mascarado    1314 non-null   str    
 12  chamado_mascarado    1319 non-null   str    
 13  descricao_mascarada  1319 non-null   str    
dtypes: float64(1), str(13)
memory usage: 144.4 KB


In [3]:
#Criando colunas com data de inicio e fim da demanda
df['inicio'] = pd.to_datetime(df['Start Date'] + ' ' + df['Start Time'], format='%d/%m/%Y %H:%M:%S')
df['fim'] = pd.to_datetime(df['End Date'] + ' ' + df['End Time'], format='%d/%m/%Y %H:%M:%S')

In [4]:
df = df.drop(['Start Date', 'Start Time', 'End Date', 'End Time'], axis=1)

In [5]:
df_ordenado = df.sort_values('Duration (decimal)', ascending=False)

demandas = df_ordenado.groupby('chamado_mascarado').agg(
    cliente=('cliente_mascarado', 'first'),
    sistema=('sistema_mascarado', 'first'),
    descricao=('descricao_mascarada', 'first'),
    tag=('Tags', 'first'),
    billable=('Billable', 'first'),
    data_inicio=('inicio', 'min'),
    data_fim=('fim', 'max'),
    esforco_horas=('Duration (decimal)', 'sum'),
    qtd_sessoes=('Duration (decimal)', 'size'),
).reset_index()

In [6]:
print(len(demandas), "demandas")
print(round(demandas['esforco_horas'].sum(), 2), "vs", round(df['Duration (decimal)'].sum(), 2))
assert round(demandas['esforco_horas'].sum(), 2) == round(df['Duration (decimal)'].sum(), 2)

385 demandas
1153.18 vs 1153.18


In [7]:
#Remove demandas sem cliente identificado
demandas = demandas[demandas['cliente'].notna()].reset_index(drop=True)

In [9]:
demandas['esforco_horas'] = demandas['esforco_horas'].round(2)

In [10]:
demandas.to_csv("demandas.csv", index=False)